# PM_system 실행

메뉴의 **런타임 → 모두 실행**을 선택하세요. CPU 런타임이면 충분하며 API 키나 추가 패키지는 필요하지 않습니다.

현재 실행 대상은 **대시보드 전체·확대 보기 SVG 시안 생성 및 미리보기**입니다. 실제 업무 관리 서버나 AI 백엔드는 아직 구현되지 않았습니다. 화면의 이름·일정·수치는 예시입니다.

[GitHub 저장소](https://github.com/phj9187/PM_system)


## 1. 저장소 준비

`main` 브랜치의 최신 코드를 가져옵니다. 같은 런타임에서 다시 실행하면 기존 폴더를 재사용합니다. 로컬 변경이나 다른 브랜치가 있으면 덮어쓰지 않고 중단합니다. 중요한 수정은 런타임 종료 전에 별도로 저장하세요.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/phj9187/PM_system.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/PM_system")

def git(*args):
    return subprocess.check_output(
        ["git", *args], text=True, stderr=subprocess.STDOUT
    ).strip()

if not PROJECT_DIR.exists():
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch",
         REPO_URL, str(PROJECT_DIR)], check=True
    )
else:
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError("같은 이름의 폴더가 이미 있습니다. PROJECT_DIR을 다른 경로로 바꾸세요.")
    actual_origin = git("-C", str(PROJECT_DIR), "remote", "get-url", "origin")
    if actual_origin.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("다른 저장소의 폴더입니다. PROJECT_DIR을 다른 경로로 바꾸세요.")
    if git("-C", str(PROJECT_DIR), "status", "--porcelain"):
        raise RuntimeError("저장되지 않은 수정이 있습니다. 수정 내용을 먼저 별도로 보관하거나 커밋하세요.")
    if git("-C", str(PROJECT_DIR), "branch", "--show-current") != BRANCH:
        raise RuntimeError("현재 브랜치가 main이 아닙니다. 작업을 보관한 뒤 main으로 전환하세요.")
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True
    )
    local = git("-C", str(PROJECT_DIR), "rev-parse", "HEAD")
    remote = git("-C", str(PROJECT_DIR), "rev-parse", "FETCH_HEAD")
    base = git("-C", str(PROJECT_DIR), "merge-base", local, remote)
    if base != local:
        raise RuntimeError("로컬 커밋이 원격과 다릅니다. 변경을 보관하고 별도 경로에서 실행하세요.")
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "merge", "--ff-only", "FETCH_HEAD"],
        check=True
    )

print("준비 완료:", PROJECT_DIR)
print("실행 버전:", git("-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"))


## 2. 대시보드 시안 생성

프로젝트의 `build_design.py`를 실행합니다. 같은 시안을 다시 생성할 수 있습니다.


In [ ]:
entrypoint = PROJECT_DIR / "build_design.py"
if not entrypoint.is_file():
    raise FileNotFoundError("build_design.py가 없습니다. 저장소 준비 단계를 확인하세요.")

subprocess.run(
    [sys.executable, str(entrypoint)],
    cwd=str(PROJECT_DIR),
    check=True
)
print("시안 생성 완료")


## 3. 전체 보기와 확대 보기

아래 화면은 정적 시안입니다. 화면에 그려진 버튼의 클릭·이동·확대 기능은 아직 구현되지 않았습니다.


In [ ]:
from IPython.display import SVG, Markdown, display

for filename, title in [
    ("01-overview.svg", "전체 보기"),
    ("02-detail.svg", "확대 보기"),
]:
    image_path = PROJECT_DIR / "designs" / filename
    if not image_path.is_file():
        raise FileNotFoundError(f"생성된 시안을 찾을 수 없습니다: {filename}")
    display(Markdown(f"### {title}"))
    display(SVG(filename=str(image_path)))

print("실행 완료. 생성 파일은 /content/PM_system/designs 에 있습니다.")
